In [0]:
%sql
DROP TABLE IF EXISTS vattenfall_dev.raw.bronze_reference;

In [0]:
import yaml
from pyspark.sql import functions as F

# ----------------------------
# 1. Load config
# ----------------------------
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]

landing_volume = config["volumes"]["landing"]
checkpoint_volume = config["volumes"]["checkpoints"]

domain = "reference"

landing_path = f"/Volumes/{catalog}/{raw_schema}/{landing_volume}/{domain}/"
checkpoint_path = f"/Volumes/{catalog}/{raw_schema}/{checkpoint_volume}/{domain}/"
schema_path = f"{checkpoint_path}/_schema"

target_table = f"{catalog}.{raw_schema}.bronze_{domain}"

print("Landing:", landing_path)
print("Checkpoint:", checkpoint_path)
print("Schema:", schema_path)
print("Table:", target_table)

# ----------------------------
# 2. Stop any running streams
# ----------------------------
for s in spark.streams.active:
    s.stop()

# ----------------------------
# 3. ONLY reset checkpoint (DO NOT delete schema)
# ----------------------------
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.mkdirs(checkpoint_path)

# ----------------------------
# 4. Check landing files (debug)
# ----------------------------
display(dbutils.fs.ls(landing_path))

# ----------------------------
# 5. Auto Loader Stream
# ----------------------------
bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(landing_path)
)

# ----------------------------
# 6. Add metadata
# ----------------------------
bronze_stream_df = (
    bronze_stream_df
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_system", F.lit(domain))
    .withColumn("source_file_path", F.col("_metadata.file_path"))
    .withColumn("source_file_name", F.col("_metadata.file_name"))
)

# ----------------------------
# 7. Write Stream (STABLE FIX)
# ----------------------------
query = (
    bronze_stream_df
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(once=True)   # 🔥 IMPORTANT FIX (ensures execution)
    .toTable(target_table)
)

query.awaitTermination()

print("✅ Bronze ingestion complete")

# ----------------------------
# 8. Verify output
# ----------------------------
spark.table(target_table).show()

In [0]:
df_test = (
    spark.read
    .option("header", True)
    .csv(landing_path)
)

df_test.show()

In [0]:
# stop running streams
for s in spark.streams.active:
    s.stop()

# clean checkpoint + schema
#dbutils.fs.rm(checkpoint_path, True)
#dbutils.fs.rm(schema_path, True)
#dbutils.fs.rm(landing_path, True)

print("Landing:", landing_path)
print("Checkpoint:", checkpoint_path)
print("Table:", target_table)
print("schema path:",schema_path)

dbutils.fs.mkdirs(landing_path)

dbutils.fs.mkdirs(checkpoint_path)
dbutils.fs.mkdirs(schema_path)

In [0]:
repo_path = f"file:/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/sample_data/{domain}"
dbutils.fs.mkdirs(landing_path)

files = dbutils.fs.ls(repo_path)



for f in files:
    try:
        dbutils.fs.cp(f.path, landing_path + f.name)
        print("Copied:", f.name)
    except Exception as e:
        print("Copy skipped or failed:", e)

display(dbutils.fs.ls(landing_path))

In [0]:


bronze_stream_df1 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(landing_path)
)

print(bronze_stream_df.columns)
#print(schema_path)
#schema_files = dbutils.fs.ls(schema_path + "/_schemas")
#latest_schema_file = sorted(schema_files, key=lambda x: x.name)[-1].path

#print(latest_schema_file)
#print(dbutils.fs.head(latest_schema_file, 5000))


In [0]:
bronze_stream_df = (
    bronze_stream_df1
      .withColumn("ingestion_ts", F.current_timestamp())
      .withColumn("source_system", F.lit(domain))
      .withColumn("source_file_path", F.col("_metadata.file_path"))
      .withColumn("source_file_name", F.col("_metadata.file_name"))
)

In [0]:
query = (
    bronze_stream_df
    .withColumn("ingestion_ts", F.current_timestamp())
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(target_table)
)


print("Bronze ingestion complete.")
#display(spark.table(target_table))
df = spark.table(target_table)
df.show()

In [0]:
query.awaitTermination()

In [0]:
display(spark.table(target_table))
